## Aegina Segmented Images (SimSiam Data) Manifest and EDA

In [4]:
%pip install pandas pillow pyarrow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import hashlib
import re

import pandas as pd
from PIL import Image


# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

DATASET_ROOT = Path(
    "/home/ars223/MSc-GEMS/IRP26/extracted_plags"
)

OUTPUT_DIRECTORY = Path(
    "/home/ars223/MSc-GEMS/IRP26/manifests"
)

CUTOFF_PX = 10

CALCULATE_HASHES = False

EXPECTED_SAMPLES = [
    "Aeg6",
    "Aeg12",
    "Aeg14",
    "Aeg20",
    "Aeg27",
    "Aeg28",
    "Aeg29",
    "Aeg30",
]


print(f"Dataset root: {DATASET_ROOT}")
print(f"Output directory: {OUTPUT_DIRECTORY}")
print(f"Dataset exists: {DATASET_ROOT.is_dir()}")

Dataset root: /home/ars223/MSc-GEMS/IRP26/extracted_plags
Output directory: /home/ars223/MSc-GEMS/IRP26/manifests
Dataset exists: True


In [6]:
FILENAME_PATTERN = re.compile(
    r"^mineral_"
    r"(?P<sample>.+)_"
    r"(?P<tile_index_1>\d+)_"
    r"(?P<tile_index_2>\d+)_"
    r"(?P<instance_index>\d+)"
    r"\.png$",
    flags=re.IGNORECASE,
)


def calculate_sha256(
    file_path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    """
    Calculate a SHA-256 hash without loading the entire file into memory.
    """
    digest = hashlib.sha256()

    with file_path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


def parse_crystal_filename(filename: str) -> dict:
    """
    Extract sample, tile indices and instance index from a PlagNet filename.

    Expected example:
        mineral_Aeg12_4_4_64.png
    """
    match = FILENAME_PATTERN.match(filename)

    if match is None:
        return {
            "filename_pattern_valid": False,
            "filename_sample": pd.NA,
            "tile_index_1": pd.NA,
            "tile_index_2": pd.NA,
            "instance_index": pd.NA,
        }

    groups = match.groupdict()

    return {
        "filename_pattern_valid": True,
        "filename_sample": groups["sample"],
        "tile_index_1": int(groups["tile_index_1"]),
        "tile_index_2": int(groups["tile_index_2"]),
        "instance_index": int(groups["instance_index"]),
    }

In [7]:
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Dataset directory does not exist: {DATASET_ROOT}"
    )

image_paths = sorted(
    DATASET_ROOT.rglob("*.png"),
    key=lambda path: path.relative_to(DATASET_ROOT).as_posix().lower(),
)

print(f"PNG files discovered: {len(image_paths):,}")

records = []

for image_number, image_path in enumerate(image_paths, start=1):
    relative_path = image_path.relative_to(DATASET_ROOT)

    # The first folder under extracted_plags is the sample name.
    sample = relative_path.parts[0]

    with Image.open(image_path) as image:
        width_px, height_px = image.size
        image_mode = image.mode
        image_format = image.format

    parsed_filename = parse_crystal_filename(image_path.name)

    minimum_dimension_px = min(width_px, height_px)
    maximum_dimension_px = max(width_px, height_px)

    geometry_path = (
        DATASET_ROOT
        / sample
        / f"{sample}_geometry.npz"
    )

    included_at_cutoff = (
        width_px >= CUTOFF_PX
        and height_px >= CUTOFF_PX
    )

    record = {
        # ----------------------------------------------------------
        # Identity and location
        # ----------------------------------------------------------
        "crystal_id": image_path.stem,
        "sample": sample,
        "filename": image_path.name,
        "relative_path": relative_path.as_posix(),

        # ----------------------------------------------------------
        # Metadata parsed from filename
        # ----------------------------------------------------------
        **parsed_filename,

        "sample_matches_filename": (
            parsed_filename["filename_sample"] == sample
            if parsed_filename["filename_pattern_valid"]
            else False
        ),

        # ----------------------------------------------------------
        # Image dimensions and derived shape information
        # ----------------------------------------------------------
        "width_px": width_px,
        "height_px": height_px,
        "minimum_dimension_px": minimum_dimension_px,
        "maximum_dimension_px": maximum_dimension_px,
        "bounding_box_area_px": width_px * height_px,
        "width_over_height": (
            width_px / height_px
            if height_px > 0
            else pd.NA
        ),
        "elongation_ratio": (
            maximum_dimension_px / minimum_dimension_px
            if minimum_dimension_px > 0
            else pd.NA
        ),
        "image_mode": image_mode,
        "image_format": image_format,

        # ----------------------------------------------------------
        # Existing PlagNet-style size cutoff
        # ----------------------------------------------------------
        "cutoff_px": CUTOFF_PX,
        "included_at_cutoff": included_at_cutoff,
        "exclusion_reason": (
            ""
            if included_at_cutoff
            else f"width_or_height_below_{CUTOFF_PX}px"
        ),

        # ----------------------------------------------------------
        # Geometry file
        # ----------------------------------------------------------
        "geometry_relative_path": (
            geometry_path.relative_to(DATASET_ROOT).as_posix()
            if geometry_path.exists()
            else pd.NA
        ),
        "geometry_file_exists": geometry_path.exists(),

        # ----------------------------------------------------------
        # File metadata
        # ----------------------------------------------------------
        "file_size_bytes": image_path.stat().st_size,
        "modified_time_utc": pd.to_datetime(
            image_path.stat().st_mtime,
            unit="s",
            utc=True,
        ),
    }

    if CALCULATE_HASHES:
        record["sha256"] = calculate_sha256(image_path)

    records.append(record)

    if image_number % 1000 == 0:
        print(
            f"Processed {image_number:,} "
            f"of {len(image_paths):,} images"
        )


manifest = pd.DataFrame.from_records(records)

print("\nManifest created.")
print(f"Shape: {manifest.shape}")

manifest.head()

PNG files discovered: 12,345
Processed 1,000 of 12,345 images
Processed 2,000 of 12,345 images
Processed 3,000 of 12,345 images
Processed 4,000 of 12,345 images
Processed 5,000 of 12,345 images
Processed 6,000 of 12,345 images
Processed 7,000 of 12,345 images
Processed 8,000 of 12,345 images
Processed 9,000 of 12,345 images
Processed 10,000 of 12,345 images
Processed 11,000 of 12,345 images
Processed 12,000 of 12,345 images

Manifest created.
Shape: (12345, 26)


,crystal_id,sample,filename,relative_path,filename_pattern_valid,filename_sample,tile_index_1,tile_index_2,instance_index,sample_matches_filename,...,elongation_ratio,image_mode,image_format,cutoff_px,included_at_cutoff,exclusion_reason,geometry_relative_path,geometry_file_exists,file_size_bytes,modified_time_utc
0,mineral_Aeg12_0_0_0,Aeg12,mineral_Aeg12_0_0_0.png,Aeg12/mineral_Aeg12_0_0_0.png,True,Aeg12,0,0,0,True,...,2.669231,RGB,PNG,10,True,,Aeg12/Aeg12_geometry.npz,True,37647,2026-04-07 03:37:26+00:00
1,mineral_Aeg12_0_0_1,Aeg12,mineral_Aeg12_0_0_1.png,Aeg12/mineral_Aeg12_0_0_1.png,True,Aeg12,0,0,1,True,...,1.031579,RGB,PNG,10,True,,Aeg12/Aeg12_geometry.npz,True,7519,2026-04-07 03:37:26+00:00
2,mineral_Aeg12_0_0_10,Aeg12,mineral_Aeg12_0_0_10.png,Aeg12/mineral_Aeg12_0_0_10.png,True,Aeg12,0,0,10,True,...,1.253165,RGB,PNG,10,True,,Aeg12/Aeg12_geometry.npz,True,5843,2026-04-07 03:37:26+00:00
3,mineral_Aeg12_0_0_11,Aeg12,mineral_Aeg12_0_0_11.png,Aeg12/mineral_Aeg12_0_0_11.png,True,Aeg12,0,0,11,True,...,1.344059,RGB,PNG,10,True,,Aeg12/Aeg12_geometry.npz,True,209605,2026-04-07 03:37:26+00:00
4,mineral_Aeg12_0_0_12,Aeg12,mineral_Aeg12_0_0_12.png,Aeg12/mineral_Aeg12_0_0_12.png,True,Aeg12,0,0,12,True,...,2.301887,RGB,PNG,10,True,,Aeg12/Aeg12_geometry.npz,True,5735,2026-04-07 03:37:26+00:00


In [ ]:
integer_columns = [
    "tile_index_1",
    "tile_index_2",
    "instance_index",
    "width_px",
    "height_px",
    "minimum_dimension_px",
    "maximum_dimension_px",
    "bounding_box_area_px",
    "cutoff_px",
    "file_size_bytes",
]

for column in integer_columns:
    manifest[column] = manifest[column].astype("Int64")


boolean_columns = [
    "filename_pattern_valid",
    "sample_matches_filename",
    "included_at_cutoff",
    "geometry_file_exists",
]

for column in boolean_columns:
    manifest[column] = manifest[column].astype("boolean")


# logical ordering for the sample column.
manifest["sample"] = pd.Categorical(
    manifest["sample"],
    categories=EXPECTED_SAMPLES,
    ordered=True,
)

manifest.dtypes

crystal_id                              object
sample                                category
filename                                object
relative_path                           object
filename_pattern_valid                 boolean
filename_sample                         object
tile_index_1                             Int64
tile_index_2                             Int64
instance_index                           Int64
sample_matches_filename                boolean
width_px                                 Int64
height_px                                Int64
minimum_dimension_px                     Int64
maximum_dimension_px                     Int64
bounding_box_area_px                     Int64
width_over_height                      float64
elongation_ratio                       float64
image_mode                              object
image_format                            object
cutoff_px                                Int64
included_at_cutoff                     boolean
exclusion_rea

In [9]:
total_images = len(manifest)

included_images = int(
    manifest["included_at_cutoff"].sum()
)

excluded_images = total_images - included_images

found_samples = [
    str(sample)
    for sample in manifest["sample"].dropna().unique()
]

missing_samples = sorted(
    set(EXPECTED_SAMPLES) - set(found_samples)
)

duplicate_crystal_ids = int(
    manifest["crystal_id"]
    .duplicated(keep=False)
    .sum()
)

duplicate_paths = int(
    manifest["relative_path"]
    .duplicated(keep=False)
    .sum()
)

invalid_filenames = int(
    (~manifest["filename_pattern_valid"]).sum()
)

sample_filename_mismatches = int(
    (~manifest["sample_matches_filename"]).sum()
)


print("PlagNet manifest validation")
print("---------------------------")
print(f"Total images:                    {total_images:,}")
print(f"Included at cutoff {CUTOFF_PX}px:        {included_images:,}")
print(f"Excluded at cutoff {CUTOFF_PX}px:             {excluded_images:,}")
print(f"Samples found:                   {found_samples}")
print(f"Missing expected samples:        {missing_samples}")
print(f"Duplicate crystal IDs:           {duplicate_crystal_ids:,}")
print(f"Duplicate relative paths:        {duplicate_paths:,}")
print(f"Invalid filename patterns:       {invalid_filenames:,}")
print(
    "Filename/sample mismatches:     "
    f"{sample_filename_mismatches:,}"
)

PlagNet manifest validation
---------------------------
Total images:                    12,345
Included at cutoff 10px:        12,344
Excluded at cutoff 10px:             1
Samples found:                   ['Aeg12', 'Aeg14', 'Aeg20', 'Aeg27', 'Aeg28', 'Aeg29', 'Aeg30', 'Aeg6']
Missing expected samples:        []
Duplicate crystal IDs:           0
Duplicate relative paths:        0
Invalid filename patterns:       0
Filename/sample mismatches:     0


In [10]:
sample_counts = (
    manifest
    .groupby("sample", observed=True)
    .agg(
        total_images=("crystal_id", "size"),
        included_images=("included_at_cutoff", "sum"),
    )
    .reset_index()
)

sample_counts["excluded_images"] = (
    sample_counts["total_images"]
    - sample_counts["included_images"]
)

sample_counts

,sample,total_images,included_images,excluded_images
0,Aeg6,1512,1512,0
1,Aeg12,2544,2544,0
2,Aeg14,1094,1094,0
3,Aeg20,2701,2701,0
4,Aeg27,1542,1541,1
5,Aeg28,765,765,0
6,Aeg29,476,476,0
7,Aeg30,1711,1711,0


In [11]:
manifest.loc[
    ~manifest["included_at_cutoff"],
    [
        "relative_path",
        "width_px",
        "height_px",
        "minimum_dimension_px",
        "exclusion_reason",
    ],
]

,relative_path,width_px,height_px,minimum_dimension_px,exclusion_reason
7854,Aeg27/mineral_Aeg27_7_4_69.png,67,7,7,width_or_height_below_10px


### This proves results are as expected - will now save the manifest for use when producing models. This 12,344-row table should become the common input for both the corrected SimSiam workflow and the new SSL (DINOV2 etc.) embeddings.

In [12]:
model_manifest = (
    manifest.loc[manifest["included_at_cutoff"]]
    .copy()
    .reset_index(drop=True)
)

print(model_manifest.shape)

(12344, 26)


In [13]:
from pathlib import Path

OUTPUT_DIRECTORY = Path(
    "/home/ars223/MSc-GEMS/IRP26/manifests"
)

model_manifest.to_csv(
    OUTPUT_DIRECTORY / "plagnet_model_manifest_cutoff10.csv",
    index=False,
)

model_manifest.to_parquet(
    OUTPUT_DIRECTORY / "plagnet_model_manifest_cutoff10.parquet",
    index=False,
)

model_manifest[["crystal_id", "sample", "relative_path"]].to_csv(
    OUTPUT_DIRECTORY / "plagnet_model_input_files_cutoff10.csv",
    index=False,
)